# Get Controls

In [ ]:
# link the databases
import os, subprocess
import pandas as pd
from google.cloud import bigquery

os.environ['GOOGLE_CLOUD_PROJECT'] = '' # you will need to add your work CDR (current directory)
os.environ['WORKSPACE_CDR'] = 'wb-silky-artichoke-2408.C2024Q3R8'
os.environ['WORKSPACE_BUCKET'] = subprocess.check_output(["wb", "resource", "resolve", "--id=WORKSPACE_BUCKET"], text=True).strip()

client = bigquery.Client(project=os.environ['GOOGLE_CLOUD_PROJECT'])
CDR = os.environ['WORKSPACE_CDR']
BUCKET = os.environ['WORKSPACE_BUCKET']

print(f"CDR: {CDR}")
print(f"BUCKET: {BUCKET}")
print("Ready.")

In [ ]:
import pandas
import os

# This query represents dataset "controls_dataset_oct_21_25" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_68024597_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` p 
            WHERE
                DATE_DIFF(CURRENT_DATE, dob, YEAR) - IF(EXTRACT(MONTH FROM dob)*100 + EXTRACT(DAY FROM dob) > EXTRACT(MONTH FROM CURRENT_DATE)*100 + EXTRACT(DAY FROM CURRENT_DATE), 1, 0) BETWEEN 60 AND 125 
                AND NOT EXISTS (      SELECT
                    'x'      
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.death` d      
                WHERE
                    d.person_id = p.person_id ) ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `""" + os.environ["WORKSPACE_CDR"] + """.person` p 
            WHERE
                race_concept_id IN (8527) ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (372605)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (1568280, 1568294, 44831122, 1568361, 1568297, 35207365, 1568284, 1568360, 35207328, 1568293, 1568299, 35207327, 44831124, 44825328, 1568298, 1568287, 1568286, 1568289, 35207329)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 )) criteria ) )"""

dataset_68024597_person_df = pandas.read_gbq(
    dataset_68024597_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_68024597_person_df.head(5)

In [ ]:
# rename df
df1 = dataset_68024597_person_df

In [ ]:
# Double check race 
df1.race.value_counts()

In [ ]:
# Check sex at birth
df1.sex_at_birth.value_counts()

In [ ]:
# Only select individuals who are male or female and are listed as white to match UKB data
sex_list = ['Male', 'Female']
df1 = df1[['person_id', 'date_of_birth', 'race', 'sex_at_birth']]
df1 = df1[df1['race']=='White']
df1 = df1[df1['sex_at_birth'].isin(sex_list)]
df1[['date_of_birth', 'extra']] = df1.date_of_birth.astype(str).str.split(' ', expand = True)
df1 = df1.drop(columns = ['extra'])
df1

In [ ]:
# Load the cases created in 01 notebookes to double check controls
import pandas as pd
cases = pd.read_csv('{path}/data/people_to_remove_from_controls.csv')
cases_list = list(cases['person_id'])
print(len(cases_list))

In [ ]:
# Only select people who were not in the cases list
df2 = df1[~df1['person_id'].isin(cases_list)]
remove = df1[df1['person_id'].isin(cases_list)]
print("controls: ", len(df2))
print("removed: ", len(remove))

In [ ]:
df2.sex_at_birth.value_counts()

In [ ]:
a = len(df2)
a

In [ ]:
# save controls
df2.to_csv(f'{path}/data/controls_60_n{a}.csv', header = True, index = False)